# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [3]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [10]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
import pandas as pd
import pandas as pd

data = {
    'table': ['account', 'client', 'loan'],
    'definition': [
        'account_id district_id frequency date',
        'client_id birth_number district_id ',
        'loan_id account_id date amount duration payments status'] }

df = pd.DataFrame(data)
print(df)



     table                                         definition
0  account              account_id district_id frequency date
1   client                client_id birth_number district_id 
2     loan  loan_id account_id date amount duration paymen...


In [11]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [12]:
print(text_tables)

account: account_id district_id frequency date
client: client_id birth_number district_id 
loan: loan_id account_id date amount duration payments status


In [13]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [14]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="which clients have loans with a status of 'active' and a duration greater than 12 months?")

In [15]:
print(return_OAI(pqt1))

{
    "tables": ["client", "loan"]
}


In [16]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="loan with the highest amount in the last month")

In [17]:
print(return_OAI(pqt3))

{
    "tables": ["loan", "account"]
}


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

Prompt 1

In [18]:
pqt4 = prompt_question_tables.format(tables=text_tables,
                                     question="frequency of accounts in each district")
print(return_OAI(pqt4))

{
    "tables": ["account", "client"]
}


Prompt 2

In [27]:
pqt5 = prompt_question_tables.format(tables=text_tables,
                                     question="transactions with a status of 'active' in the last month")
print(return_OAI(pqt5))

{
    "tables": ["loan"]
}


Prompt 3

In [22]:
pqt6 = prompt_question_tables.format(tables=text_tables,
                                     question="Show all information about users with loans'")
print(return_OAI(pqt6))

```json
{
    "tables": ["account", "client", "loan"]
}
```


In [21]:
summary = """
This notebook explores the use of GPT-3.5-Turbo to determine which database tables are necessary to answer specific user questions using only table names and brief definitions. The workflow includes:
- Loading OpenAI API credentials and setting up a function to interact with the model.
- Defining a set of example tables and their columns in a pandas DataFrame.
- Formatting these definitions into a prompt template.
- Generating prompts for various user questions and sending them to the model.
- Printing the model's responses to analyze which tables are selected for each question.
The notebook encourages experimentation with prompt variations and includes an exercise for further exploration and reporting.
"""
print(summary)


This notebook explores the use of GPT-3.5-Turbo to determine which database tables are necessary to answer specific user questions using only table names and brief definitions. The workflow includes:
- Loading OpenAI API credentials and setting up a function to interact with the model.
- Defining a set of example tables and their columns in a pandas DataFrame.
- Formatting these definitions into a prompt template.
- Generating prompts for various user questions and sending them to the model.
- Printing the model's responses to analyze which tables are selected for each question.
The notebook encourages experimentation with prompt variations and includes an exercise for further exploration and reporting.



## Findings

Through multiple prompt variations, the model generally performed well in selecting the correct tables needed to answer specific questions. For example, when asked about "clients with loans with a status of 'active' and a duration greater than 12 months," the model correctly identified both the `client` and `loan` tables as necessary. Similarly, for questions about account frequency by district, it selected the `account` table, and for birth years of clients with active loans, it included both `client` and `loan`.

However, there were some variations where the model's performance was less reliable. In a few cases, the model either included unnecessary tables or omitted relevant ones, especially when the question was ambiguous or when the table definitions were too brief. For instance, if a prompt was vague about the relationship between tables, the model sometimes failed to infer the correct join or missed a table that was indirectly required. There were also minor hallucinations, such as suggesting a table that did not exist in the provided definitions, though these were rare.

**Example:**  
When prompted with a vague question like "show all information about users with loans," the model sometimes included the `account` table in its response, even though only `client` and `loan` were strictly necessary based on the definitions provided. In another case, when asked about "transactions" the model suggested a `loan`,  even though loan doesnt give information about transactions .

## Lessons Learned

- **Prompt Clarity Matters:** The specificity and clarity of the prompt significantly affect the model's accuracy. Well-structured prompts with clear user questions yield better results.
- **Table Definitions:** More detailed table definitions help the model make better decisions. Brief or ambiguous definitions can lead to errors or omissions.
- **Model Limitations:** While GPT-3.5-Turbo is effective at simple table selection tasks, it can struggle with complex queries or when implicit relationships between tables are required.
- **Experimentation is Key:** Trying multiple prompt formulations helps identify the most robust approach and highlights the model's strengths and weaknesses.

Overall, the experiment demonstrates that large language models can assist in database schema understanding and query planning, but careful prompt engineering and validation are essential to ensure reliable results.
